In [1]:
from collections import defaultdict
import networkx as nx
import pandas as pd
import numpy as np
import json

G = nx.read_gml('../file_merging/Updated/network_graph_weighted.gml')

In [2]:
## Remove isolated variants

isolated_nodes = [
    node
    for node, degree in G.degree()
    if degree == 0 #and G.nodes[node].get("category") == "Variant"
]

print("Number of nodes with no edges:", len(isolated_nodes))

G.remove_nodes_from(isolated_nodes)
print("New node count:", G.number_of_nodes())

remaining_isolates = list(nx.isolates(G))
print("Remaining isolated nodes:", len(remaining_isolates))

# Save the graph without isolated variants
nx.write_gml(G, '../file_merging/Updated/network_graph_weighted.gml')

Number of nodes with no edges: 0
New node count: 4388
Remaining isolated nodes: 0


In [3]:
category_sets = defaultdict(set)

for node, data in G.nodes(data=True):
    category = data.get("category")
    if category:
        category_sets[category].add(node)

print("Variants:", len(category_sets["Variant"]))
print("Cancers:", len(category_sets["Cancer"]))
print("Treatments:", len(category_sets["Treatment"]))

from collections import Counter

Counter(d.get("category") for _, d in G.nodes(data=True))

Variants: 3902
Cancers: 98
Treatments: 388


Counter({'Variant': 3902, 'Treatment': 388, 'Cancer': 98})

In [4]:
sentence = (
    f"represented in a literature-derived graph with "
    f"{G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges."
)

print(sentence)

represented in a literature-derived graph with 4,388 nodes and 46,943 edges.


In [16]:
import numpy as np

# Extract weights safely
weights = [
    d["weight"]
    for _, _, d in G.edges(data=True)
    if "weight" in d
]

threshold = np.percentile(weights, 80)

significant_edges = [
    (u, v)
    for u, v, d in G.edges(data=True)
    if d.get("weight", float("-inf")) > threshold
]

print("80th percentile threshold:", threshold)
print("Significant edges:", len(significant_edges))

80th percentile threshold: 1.7
Significant edges: 9155


## Graph use

In [14]:
import networkx as nx
import pandas as pd
import numpy as np
import json

G = nx.read_gml('../file_merging/Updated/network_graph_weighted.gml')

######## UPDATED

#### Updated weighted network graph with automated threshold, based on qualitative analysis

variant_of_interest = "l858r_EGFR"
cancer_of_interest = "lung cancer"

# Adjustable thresholds
TREATMENT_THRESHOLD_PERCENTILE = 80    # highlight top X% of treatment weights
TREATMENT_MIN_HIGHLIGHT        = 300   # and require ≥X total weight
CANCER_THRESHOLD_PERCENTILE    = 80    # highlight top X% of cancer–variant weights
CANCER_MIN_HIGHLIGHT           = 80    # and require ≥X total weight

df_consensus = pd.read_csv("../file_merging/Updated/final_variant_treatment_consensus.csv")

# Prepare consensus lookup
df_consensus["Variant_Treatment_Pair"] = (
    df_consensus["Variant_Treatment_Pair"]
    .str.strip()
    .str.lower()
)
consensus_dict = dict(
    zip(df_consensus["Variant_Treatment_Pair"], df_consensus["Resolved_Prediction"])
)

excluded_treatments = {
    'chemotherapy', 'tyrosine kinase inhibitor', 'radiotherapy', 'hormone therapy',
    'adjuvant chemotherapy', 'immunotherapy', 'immune checkpoint inhibitor', 'adjuvant chemotherapy',
    'mrna vaccine', 'mtor inhibitor', 'radiation ionizing radiotherapy', 
    'braf inhibitor','angiogenesis inhibitor', 'aromatase inhibitor', 'bet inhibitor',
    'EGFR tyrosine kinase inhibitor therapy', 'epidermal growth factor receptor tyrosine kinase inhibitor',
    'hematopoietic cell transplantation', 'hyperthermic intraperitoneal chemotherapy', 'TRK inhibitor',
    'tyrosine kinase inhibitor', 'therapeutic tumor infiltrating lymphocytes'
}

# Cancer‐only treatments
canc_nei = set(G.neighbors(cancer_of_interest))
treatments = [
    n for n in canc_nei
    if G.nodes[n]['category']=='Treatment'
    and n.lower() not in excluded_treatments
]
t_weights = {t: G[cancer_of_interest][t]['weight'] for t in treatments}
top_cancer_treats = sorted(t_weights.items(), key=lambda x: x[1], reverse=True)[:6]
c_w = list(t_weights.values())
treat_pct = np.percentile(c_w, TREATMENT_THRESHOLD_PERCENTILE) if c_w else 0


# Variant + cancer associations
sensitive, resistant = [], []
for t in treatments:
    try:
        w = G[cancer_of_interest][t]['weight'] + G[variant_of_interest][t]['weight']
        pred = consensus_dict.get(f"{variant_of_interest} + {t}".lower())
        if pred == "Sensitive":
            sensitive.append((t, w))
        elif pred == "Resistant":
            resistant.append((t, w))
    except KeyError:
        continue

top_sens = sorted(sensitive, key=lambda x: x[1], reverse=True)[:6]
top_res  = sorted(resistant, key=lambda x: x[1], reverse=True)[:6]
sens_w = [w for _, w in sensitive]
res_w  = [w for _, w in resistant]
sens_pct = np.percentile(sens_w, TREATMENT_THRESHOLD_PERCENTILE) if sens_w else 0
res_pct  = np.percentile(res_w,   TREATMENT_THRESHOLD_PERCENTILE) if res_w else 0

sens_strong_json = []
sens_weak_json = []
res_strong_json = []
res_weak_json = []

print(f"\n\033[1mSensitive treatments for variant '{variant_of_interest}' "
      f"(≥{TREATMENT_THRESHOLD_PERCENTILE}th pct & ≥{TREATMENT_MIN_HIGHLIGHT}):\033[0m")
for t, w in top_sens:
    if w >= sens_pct and w >= TREATMENT_MIN_HIGHLIGHT:
        sens_strong_json.append({"treatment": t, "weight": round(w)})
        print(f"\033[1;32m{t}: {w:.0f}\033[0m")
    else:
        sens_weak_json.append({"treatment": t, "weight": round(w)})
        print(f"\033[2;37m{t}: {w:.0f}\033[0m")

print(f"\n\033[1mResistant treatments for variant '{variant_of_interest}' "
      f"(≥{TREATMENT_THRESHOLD_PERCENTILE}th pct & ≥{TREATMENT_MIN_HIGHLIGHT}):\033[0m")
for t, w in top_res:
    if w >= res_pct and w >= TREATMENT_MIN_HIGHLIGHT:
        res_strong_json.append({"treatment": t, "weight": round(w)})
        print(f"\033[1;31m{t}: {w:.0f}\033[0m")
    else:
        res_weak_json.append({"treatment": t, "weight": round(w)})
        print(f"\033[2;37m{t}: {w:.0f}\033[0m")

df = pd.DataFrame({
    "variant": variant_of_interest,
    "cancer": cancer_of_interest,
    "sensitive_treatments_strong": json.dumps(sens_strong_json),
    "sensitive_treatments_weak": json.dumps(sens_weak_json),
    "resistant_treatments_strong": json.dumps(res_strong_json),
    "resistant_treatments_weak": json.dumps(res_weak_json),
}, index=[0])
df


Sensitive treatments for variant 'l858r_EGFR' (≥80th pct & ≥300):
Osimertinib: 678
Gefitinib: 477
Erlotinib: 430
Afatinib: 333
Radiation Therapy: 169
Crizotinib: 155

Resistant treatments for variant 'l858r_EGFR' (≥80th pct & ≥300):
Cisplatin: 179
Pembrolizumab: 46
Paclitaxel: 30
Docetaxel: 30
Nivolumab: 22
Lapatinib: 17


,variant,cancer,sensitive_treatments_strong,sensitive_treatments_weak,resistant_treatments_strong,resistant_treatments_weak
0,l858r_EGFR,lung cancer,"[{""treatment"": ""Osimertinib"", ""weight"": 678}, ...","[{""treatment"": ""Radiation Therapy"", ""weight"": ...",[],"[{""treatment"": ""Cisplatin"", ""weight"": 179}, {""..."


In [16]:
variant = "l858r_EGFR"

cancers = [
    neighbor
    for neighbor in G.neighbors(variant)
    if G.nodes[neighbor].get("category") == "Cancer"
]

print(cancers)
print("Total cancers:", len(cancers))

['lung cancer', 'breast cancer', 'glioblastoma', 'head and neck cancer', 'colon cancer', 'squamous cell cancer', 'leukemia', 'pancreatic cancer', 'glioma', 'neuroblastoma', 'bladder urothelial cancer', 'urothelial cancer', 'myeloid cancer', 'thyroid cancer', 'ovarian cancer', 'bladder cancer', 'cervix cancer', 'neuroendocrine cancer', 'biliary tract cancer', 'cholangiocarcinoma', 'gallbladder cancer', 'gastrointestinal cancer', 'uterine cancer', 'liver cancer', 'acute myeloid leukemia', 'melanoma', 'renal cancer']
Total cancers: 27
